Ground truth log:

In [ ]:
import subprocess, sys, json

_CHILD = r'''
import sys, json, tkinter as tk
from tkinter import filedialog
opts = json.loads(sys.argv[1])
root = tk.Tk(); root.withdraw()
try: root.attributes("-topmost", True)
except Exception: pass
p = filedialog.askopenfilename(title=opts["title"], initialdir=opts["initialdir"],
                               filetypes=[tuple(ft) for ft in opts["filetypes"]])
root.destroy()
sys.stdout.write(p or "")
'''

def pick_file(title="Select a file", initialdir=".",
              filetypes=(("OCEL logs", "*.json *.jsonocel"), ("All files", "*.*"))):
    """Runs the Tk dialog in a child process: if Tk dies, only the child dies."""
    opts = {"title": title, "initialdir": initialdir,
            "filetypes": [list(ft) for ft in filetypes]}
    r = subprocess.run([sys.executable, "-c", _CHILD, json.dumps(opts)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip() or f"file picker exited {r.returncode}")
    return r.stdout.strip() or None

def ensure_ocel2(path):
    """Normalize OCEL 1.0 to the OCEL 2.0-shaped JSON used by the evaluation.

    OCEL 2.0 inputs are returned unchanged. For OCEL 1.0, write a unique
    temporary copy; the original is never modified. Preserve the activity,
    timestamp, object type and event-object links used by these measures.
    Attributes are omitted; OCEL 1.0 has no object-to-object relationships.
    """
    from pathlib import Path
    import tempfile

    path = Path(path)
    with path.open(encoding="utf-8") as f:
        raw = json.load(f)
    if isinstance(raw.get("events"), list) and isinstance(raw.get("objects"), list):
        print(f"OCEL 2.0: {path.name}")
        return str(path)
    if not isinstance(raw.get("ocel:events"), dict) or not isinstance(raw.get("ocel:objects"), dict):
        raise ValueError(f"{path.name} is neither supported OCEL 2.0 JSON nor OCEL 1.0 JSON-OCEL")

    objects = [{"id": str(oid), "type": obj["ocel:type"], "attributes": []}
               for oid, obj in raw["ocel:objects"].items()]
    events = [{"id": str(eid), "type": ev["ocel:activity"],
               "time": ev["ocel:timestamp"], "attributes": [],
               "relationships": [{"objectId": str(oid), "qualifier": ""}
                                 for oid in (ev.get("ocel:omap") or [])]}
              for eid, ev in raw["ocel:events"].items()]
    converted = {
        "objectTypes": [{"name": t, "attributes": []}
                        for t in sorted({o["type"] for o in objects})],
        "eventTypes": [{"name": a, "attributes": []}
                       for a in sorted({e["type"] for e in events})],
        "objects": objects, "events": events,
    }
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", suffix="_ocel2.json",
                                     prefix="ocdeclare_eval_", delete=False) as f:
        json.dump(converted, f, ensure_ascii=False)
        converted_path = f.name
    print(f"OCEL 1.0: {path.name} -> {converted_path}")
    print(f"Converted {len(events):,} events / {len(objects):,} objects")
    return converted_path

INPUT_LOG = pick_file("Select the input (ground truth) log — OCEL 2.0 or OCEL 1.0")
if INPUT_LOG is None:
    raise SystemExit("No log selected")
INPUT_LOG = ensure_ocel2(INPUT_LOG)

# Log-derived caches must not survive a new file selection.
for _cache_name in ("log_in_red", "log_sim_red", "log_in_ngd", "log_sim_ngd", "TEMPORAL"):
    globals().pop(_cache_name, None)


Simulated event log:

In [ ]:
import subprocess, sys, json

_CHILD = r'''
import sys, json, tkinter as tk
from tkinter import filedialog
opts = json.loads(sys.argv[1])
root = tk.Tk(); root.withdraw()
try: root.attributes("-topmost", True)
except Exception: pass
p = filedialog.askopenfilename(title=opts["title"], initialdir=opts["initialdir"],
                               filetypes=[tuple(ft) for ft in opts["filetypes"]])
root.destroy()
sys.stdout.write(p or "")
'''

def pick_file(title="Select a file", initialdir=".",
              filetypes=(("OCEL logs", "*.json *.jsonocel"), ("All files", "*.*"))):
    """Runs the Tk dialog in a child process: if Tk dies, only the child dies."""
    opts = {"title": title, "initialdir": initialdir,
            "filetypes": [list(ft) for ft in filetypes]}
    r = subprocess.run([sys.executable, "-c", _CHILD, json.dumps(opts)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(r.stderr.strip() or f"file picker exited {r.returncode}")
    return r.stdout.strip() or None

SIMULATED_LOG = pick_file("Select the simulated log — OCEL 2.0 or OCEL 1.0")
if SIMULATED_LOG is None:
    raise SystemExit("No log selected")
SIMULATED_LOG = ensure_ocel2(SIMULATED_LOG)

# Log-derived caches must not survive a new file selection.
for _cache_name in ("log_in_red", "log_sim_red", "log_in_ngd", "log_sim_ngd", "TEMPORAL"):
    globals().pop(_cache_name, None)


OC-Declare Model file:

In [28]:
# ===== Select the OC-Declare model ==========================================
import json
from pathlib import Path

# start in the ocdeclare folder next to the chosen input log, if it exists
_default = Path(INPUT_LOG).parent.parent / "ocdeclare"
OCDECLARE_MODEL = pick_file("Select the OC-Declare model",
                            initialdir=str(_default if _default.exists() else Path(INPUT_LOG).parent),
                            filetypes=(("JSON files", "*.json"), ("All files", "*.*")))
if OCDECLARE_MODEL is None:
    raise SystemExit("no model selected")

def load_model(path, expect_activities=None):
    """Load an OC-Declare model and fail loudly if the file is actually an OCEL log —
       both are .json and they sit in sibling folders, so this is an easy mix-up."""
    path = Path(path)
    d = json.load(open(path))
    if "events" in d and "objects" in d:
        raise ValueError(f"{path.name} is an OCEL event log, not an OC-Declare model")
    if "constraints" not in d or "activities" not in d:
        raise ValueError(f"{path.name} has no 'constraints'/'activities' — not an OC-Declare model "
                         f"(top-level keys: {sorted(d)[:8]})")
    acts = sorted(a["name"] for a in d["activities"])
    print(f"model: {path.name}")
    print(f"  {len(d['constraints'])} constraints, {len(acts)} activities, "
          f"{len(d.get('object_types', []))} object types, {len(d.get('o2o_rules', []))} o2o rules")
    if expect_activities is not None:
        missing = set(acts) - set(expect_activities)
        if missing: print(f"  ! model activities absent from the input log: {sorted(missing)}")
        else:       print("  activity alphabet matches the input log")
    return d

model = load_model(OCDECLARE_MODEL)
# after log_in exists, you can check the pairing:
# model = load_model(OCDECLARE_MODEL, expect_activities={a for _, a, _, _ in log_in["events"]})

model: discovered_container_logistics_20260924_093424.json
  35 constraints, 14 activities, 7 object types, 0 o2o rules


In [29]:
# ===== Load the three inputs once (run after the file-selection cell) =======
import json, pandas as pd

def load_ocel(path):
    raw = json.load(open(path))
    ts = pd.to_datetime([e["time"] for e in raw["events"]], utc=True, format="ISO8601")
    ev = sorted(zip((e["id"] for e in raw["events"]), (e["type"] for e in raw["events"]), ts,
                    ([r["objectId"] for r in e.get("relationships") or []] for e in raw["events"])),
                key=lambda x: x[2])
    return {"events": ev, "otype": {o["id"]: o["type"] for o in raw["objects"]}, "raw": raw}

log_in  = load_ocel(INPUT_LOG)        # parsed OCEL, what every measure consumes
log_sim = load_ocel(SIMULATED_LOG)
model   = json.load(open(OCDECLARE_MODEL))

print(f"input:     {len(log_in['events']):>7,} events, {len(log_in['otype']):>6,} objects")
print(f"simulated: {len(log_sim['events']):>7,} events, {len(log_sim['otype']):>6,} objects")
print(f"model:     {len(model['constraints'])} constraints, {len(model['activities'])} activities")

input:      35,372 events, 13,882 objects
simulated:     362 events,    185 objects
model:     35 constraints, 14 activities


Simulated log difference from ground truth, events and objects:

KL Divergence events

In [30]:
# ============ M1 — KL divergence for activities (log comparison) ==============
# Requires INPUT_LOG and SIMULATED_LOG from the input-files cell.
# KL is calculated with additive smoothing; JS is not calculated.

import json
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

SMOOTHING = 0.5


def load_ocel(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    ts = pd.to_datetime(
        [e["time"] for e in raw["events"]],
        utc=True,
        format="ISO8601",
    )

    events = sorted(
        zip(
            (e["id"] for e in raw["events"]),
            (e["type"] for e in raw["events"]),
            ts,
            (
                [r["objectId"] for r in e.get("relationships") or []]
                for e in raw["events"]
            ),
        ),
        key=lambda event: event[2],
    )

    return {
        "events": events,
        "otype": {o["id"]: o["type"] for o in raw["objects"]},
        "raw": raw,
    }


def smoothed_probabilities(cP, cQ, smoothing=SMOOTHING):
    """Align categories and normalize each histogram with additive smoothing."""
    if not np.isfinite(smoothing) or smoothing <= 0:
        raise ValueError("smoothing must be a finite number greater than zero.")

    keys = sorted(set(cP) | set(cQ))
    counts_p = np.array([cP.get(k, 0) for k in keys], dtype=float)
    counts_q = np.array([cQ.get(k, 0) for k in keys], dtype=float)

    for name, counts in (("P", counts_p), ("Q", counts_q)):
        if not np.all(np.isfinite(counts)) or np.any(counts < 0):
            raise ValueError(f"{name} must contain finite, nonnegative counts.")
        if counts.sum() <= 0:
            raise ValueError(
                f"{name} contains no observations; KL divergence is undefined."
            )

    p = counts_p + smoothing
    q = counts_q + smoothing
    p /= p.sum()
    q /= q.sum()

    return keys, p, q


def kl_divergence(cP, cQ, smoothing=SMOOTHING):
    """Return D_KL(P || Q) in bits for additively smoothed distributions."""
    _, p, q = smoothed_probabilities(cP, cQ, smoothing)
    return float(np.sum(p * np.log2(p / q)))


log_in = load_ocel(INPUT_LOG)
log_sim = load_ocel(SIMULATED_LOG)

c_in = Counter(activity for _, activity, _, _ in log_in["events"])
c_sim = Counter(activity for _, activity, _, _ in log_sim["events"])

# Use these same probabilities for both headline scores and contributions.
activities, p, q = smoothed_probabilities(c_in, c_sim)

contributions_forward = p * np.log2(p / q)
contributions_reverse = q * np.log2(q / p)

kl_activities = {
    "n_activities": len(activities),
    "smoothing": SMOOTHING,
    "KL_input||simulated_bits": float(contributions_forward.sum()),
    "KL_simulated||input_bits": float(contributions_reverse.sum()),
    "only_in_input": sorted(set(c_in) - set(c_sim)),
    "only_in_simulated": sorted(set(c_sim) - set(c_in)),
}

n_input = sum(c_in.values())
n_simulated = sum(c_sim.values())

per_activity = pd.DataFrame({
    "activity": activities,
    "count_input": [c_in.get(a, 0) for a in activities],
    "count_simulated": [c_sim.get(a, 0) for a in activities],
    "p_input_raw": [c_in.get(a, 0) / n_input for a in activities],
    "p_simulated_raw": [c_sim.get(a, 0) / n_simulated for a in activities],
    "p_input_smoothed": p,
    "p_simulated_smoothed": q,
    "contribution_bits": contributions_forward,
})

print(f"Activity KL divergence — additive smoothing α = {SMOOTHING}")
print(
    f"KL(input || simulated) = "
    f"{kl_activities['KL_input||simulated_bits']:.4f} bits"
)
print(
    f"KL(simulated || input) = "
    f"{kl_activities['KL_simulated||input_bits']:.4f} bits"
)
print(f"Events: input = {n_input:,}; simulated = {n_simulated:,}")
print(
    f"{kl_activities['n_activities']} activities | "
    f"only in input: {kl_activities['only_in_input']} | "
    f"only in simulated: {kl_activities['only_in_simulated']}"
)
print(
    "Per-activity contributions sum to KL(input || simulated). "
    "Individual contributions may be negative."
)

display(
    per_activity.sort_values("contribution_bits", ascending=False)
    .reset_index(drop=True)
)

Activity KL divergence — additive smoothing α = 0.5
KL(input || simulated) = 0.0556 bits
KL(simulated || input) = 0.1364 bits
Events: input = 35,372; simulated = 362
14 activities | only in input: ['Depart'] | only in simulated: []
Per-activity contributions sum to KL(input || simulated). Individual contributions may be negative.


,activity,count_input,count_simulated,p_input_raw,p_simulated_raw,p_input_smoothed,p_simulated_smoothed,contribution_bits
0,Load Truck,10553,94,0.298343,0.259669,0.298298,0.256098,0.065644
1,Collect Goods,10553,105,0.298343,0.290055,0.298298,0.285908,0.018258
2,Place in Stock,1814,16,0.051284,0.044199,0.051287,0.044715,0.010146
3,Depart,127,0,0.003590,0.000000,0.003604,0.001355,0.005086
4,Bring to Loading Bay,1969,19,0.055665,0.052486,0.055669,0.052846,0.004180
5,Drive to Terminal,1989,20,0.056231,0.055249,0.056234,0.055556,0.000985
6,Weigh,1989,20,0.056231,0.055249,0.056234,0.055556,0.000985
7,Register Customer Order,600,6,0.016963,0.016575,0.016973,0.017615,-0.000909
8,Book Vehicles,594,6,0.016793,0.016575,0.016804,0.017615,-0.001143
9,Create Transport Document,594,6,0.016793,0.016575,0.016804,0.017615,-0.001143


KL Divergence for object types:

In [31]:
# ============ M2 — KL divergence for object types (log comparison) ============
# Run the corrected M1 cell first.
# Requires: load_ocel, smoothed_probabilities, SMOOTHING,
#           INPUT_LOG and SIMULATED_LOG.

from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display


def type_counts(log, basis):
    """
    basis='objects':
        Count objects of each type, including objects without events.

    basis='relations':
        Count event-to-object attachments by object type.
        An event involving five distinct objects of one type contributes five.
        Repeated references to the same object within an event count once,
        regardless of relationship qualifier.
    """
    if basis == "objects":
        return Counter(log["otype"].values())

    if basis != "relations":
        raise ValueError("basis must be 'objects' or 'relations'.")

    counts = Counter()

    for event_id, _, _, object_ids in log["events"]:
        for object_id in set(object_ids):
            if object_id not in log["otype"]:
                raise ValueError(
                    f"Event {event_id!r} references unknown object "
                    f"{object_id!r}."
                )
            counts[log["otype"][object_id]] += 1

    return counts


A = load_ocel(INPUT_LOG)
B = load_ocel(SIMULATED_LOG)

summary_rows = []
detail_tables = {}

for basis in ("objects", "relations"):
    cA = type_counts(A, basis)
    cB = type_counts(B, basis)

    # Rejects empty histograms and invalid smoothing/counts.
    try:
        types, p, q = smoothed_probabilities(
            cA, cB, smoothing=SMOOTHING
        )
    except ValueError as exc:
        raise ValueError(f"Object-type KL, basis={basis!r}: {exc}") from exc

    contributions_forward = p * np.log2(p / q)
    contributions_reverse = q * np.log2(q / p)

    nA = sum(cA.values())
    nB = sum(cB.values())

    summary_rows.append({
        "basis": basis,
        "n_types": len(types),
        "n_input": nA,
        "n_simulated": nB,
        "smoothing": SMOOTHING,
        "KL_input||simulated_bits": float(contributions_forward.sum()),
        "KL_simulated||input_bits": float(contributions_reverse.sum()),
        "only_in_input": sorted(set(cA) - set(cB)),
        "only_in_simulated": sorted(set(cB) - set(cA)),
    })

    detail_tables[basis] = pd.DataFrame({
        "object_type": types,
        "count_input": [cA.get(t, 0) for t in types],
        "count_simulated": [cB.get(t, 0) for t in types],
        "p_input_raw": [cA.get(t, 0) / nA for t in types],
        "p_simulated_raw": [cB.get(t, 0) / nB for t in types],
        "p_input_smoothed": p,
        "p_simulated_smoothed": q,
        "contribution_bits": contributions_forward,
    }).sort_values(
        "contribution_bits", ascending=False
    ).reset_index(drop=True)


kl_object_types = pd.DataFrame(summary_rows)

# Separate detail tables for the two comparison perspectives.
per_type = detail_tables["objects"]
per_type_relations = detail_tables["relations"]

print(f"Object-type KL divergence — additive smoothing α = {SMOOTHING}")
display(kl_object_types)





Object-type KL divergence — additive smoothing α = 0.5


,basis,n_types,n_input,n_simulated,smoothing,KL_input||simulated_bits,KL_simulated||input_bits,only_in_input,only_in_simulated
0,objects,7,13882,185,0.5,0.349579,1.493493,[],[]
1,relations,7,74272,738,0.5,0.008231,0.009107,[],[]


Time comparison:

Time distribution on log

In [35]:
# ===== Discover estimated service-time parameters per activity ===============
# Requires INPUT_LOG and SIMULATED_LOG.
#
# Matches the prototype's service-time estimation:
# 1. Collect forward gaps per event-object attachment.
# 2. Collect sojourn times per event.
# 3. Choose the source with the smaller middle observation.
# 4. Select the configured duration window.
# 5. Calculate its mean and population standard deviation.
#
# These are heuristic service-time estimates, not measured processing durations.

import json
import math
import statistics
from collections import defaultdict

import pandas as pd
from IPython.display import display

SERVICE_TIME_MODE = "minimum"   # "minimum", "p25", or "p50"
ANCHOR_ACTIVITIES = []          # Optional prototype-style parameter overrides


def discover_service_times(
    log_path,
    service_time_mode="minimum",
    anchor_activities=None,
):
    if service_time_mode not in {"minimum", "p25", "p50"}:
        raise ValueError("service_time_mode must be 'minimum', 'p25', or 'p50'.")

    with open(log_path, encoding="utf-8") as f:
        raw = json.load(f)

    events = raw["events"]
    if not isinstance(events, list):
        raise ValueError("Expected an OCEL 2.0 event list.")

    timestamps = pd.to_datetime(
        [event.get("time") for event in events],
        utc=True,
        format="ISO8601",
        errors="raise",
    )
    if timestamps.isna().any():
        raise ValueError("Every event must have a valid timestamp.")

    known_objects = {obj["id"] for obj in raw["objects"]}
    timelines = defaultdict(list)
    event_objects = []

    for i, event in enumerate(events):
        object_ids = [
            relation["objectId"]
            for relation in event.get("relationships") or []
        ]

        # Count each event-object pair once, ignoring relationship qualifiers.
        object_ids = list(dict.fromkeys(object_ids))
        event_objects.append(object_ids)

        for object_id in object_ids:
            if object_id not in known_objects:
                raise ValueError(
                    f"Event {event['id']!r} references unknown object "
                    f"{object_id!r}."
                )
            timelines[object_id].append(i)

    preceding = defaultdict(list)
    following = defaultdict(list)

    for timeline in timelines.values():
        # Stable sorting preserves input order for equal timestamps.
        timeline.sort(key=lambda i: timestamps[i])

        for previous_i, next_i in zip(timeline, timeline[1:]):
            following[previous_i].append(timestamps[next_i])
            preceding[next_i].append(timestamps[previous_i])

    buckets = defaultdict(lambda: {"forward": [], "sojourn": []})

    for i, event in enumerate(events):
        activity = event.get("type")
        if not activity or not event_objects[i]:
            continue

        complete = timestamps[i]
        bucket = buckets[activity]

        # One forward observation per object that has a subsequent event.
        bucket["forward"].extend(
            max(0.0, (next_time - complete).total_seconds())
            for next_time in following[i]
        )

        # One sojourn observation per event.
        # Match the prototype's zero fallback when no predecessor exists.
        sojourn = (
            max(
                0.0,
                (complete - max(preceding[i])).total_seconds(),
            )
            if preceding[i]
            else 0.0
        )
        bucket["sojourn"].append(sojourn)

    anchors = {
        anchor["name"]: anchor
        for anchor in anchor_activities or []
        if anchor.get("name")
    }

    def summarize(values):
        if not values:
            return {"mean": 0.0, "std": 0.0, "min": 0.0, "max": 0.0}

        return {
            "mean": float(statistics.mean(values)),
            # Prototype uses population std, not sample std.
            "std": float(statistics.pstdev(values)),
            "min": float(min(values)),
            "max": float(max(values)),
        }

    results = {}

    for activity in sorted(set(buckets) | set(anchors)):
        bucket = buckets.get(activity, {"forward": [], "sojourn": []})
        direct = bucket["forward"]
        sojourn = bucket["sojourn"]
        positive_sojourn = [value for value in sojourn if value > 0]

        # Match the prototype's upper-middle observation for even sample sizes.
        if direct and positive_sojourn:
            direct_middle = sorted(direct)[len(direct) // 2]
            sojourn_middle = sorted(positive_sojourn)[len(positive_sojourn) // 2]

            if direct_middle <= sojourn_middle:
                source, source_name = direct, "forward"
            else:
                source, source_name = positive_sojourn, "positive_sojourn"
        elif direct:
            source, source_name = direct, "forward"
        elif positive_sojourn:
            source, source_name = positive_sojourn, "positive_sojourn"
        else:
            source, source_name = sojourn, "zero_or_empty_sojourn"

        selected_count = 0
        selected = []   # Anchors and unavailable estimates have no empirical samples.

        if activity in anchors:
            anchor = anchors[activity]
            sojourn_stats = summarize(sojourn)

            mean = float(anchor.get("mean_seconds", sojourn_stats["mean"]))
            std = float(anchor.get("std_seconds", sojourn_stats["std"]))
            minimum = float(anchor.get("min_seconds", 0.0))
            maximum = anchor.get("max_seconds")
            maximum = float(maximum) if maximum is not None else None

            source_name = "anchor"
            window = "anchor"

        elif source:
            ordered = sorted(source)
            n = len(ordered)

            if service_time_mode == "p25":
                lower = 0
                upper = max(0, math.ceil(0.50 * n) - 1)
            elif service_time_mode == "p50":
                lower = max(0, math.ceil(0.25 * n) - 1)
                upper = max(0, math.ceil(0.75 * n) - 1)
            else:
                lower = 0
                upper = max(0, math.ceil(0.25 * n) - 1)

            selected = ordered[lower:upper + 1]
            stats = summarize(selected)
            selected_count = len(selected)

            mean, std = stats["mean"], stats["std"]
            minimum, maximum = stats["min"], stats["max"]
            window = service_time_mode

        else:
            mean, std, minimum, maximum = 0.0, 0.0, 0.0, None
            window = "no_observations"

        results[activity] = {
            "service_mean": mean,     # Seconds
            "service_std": std,       # Seconds; population standard deviation
            "service_min": minimum,   # Seconds
            "service_max": maximum,   # Seconds
            "estimation_source": source_name,
            "estimation_window": window,
            "n_forward_observations": len(direct),
            "n_sojourn_observations": len(sojourn),
            "n_selected_observations": selected_count,
            "service_samples_seconds": list(selected),
        }

    return results


# Store both sets of discovered activity-level parameters for later cells.
SERVICE_TIMES = {
    "input": discover_service_times(
        INPUT_LOG, SERVICE_TIME_MODE, ANCHOR_ACTIVITIES
    ),
    "simulated": discover_service_times(
        SIMULATED_LOG, SERVICE_TIME_MODE, ANCHOR_ACTIVITIES
    ),
}

for role, activities in SERVICE_TIMES.items():
    print(f"\n{role}: {len(activities)} activities — durations in seconds")
    table = pd.DataFrame.from_dict(activities, orient="index")
    table.index.name = "activity"
    display(table)


input: 14 activities — durations in seconds


,service_mean,service_std,service_min,service_max,estimation_source,estimation_window,n_forward_observations,n_sojourn_observations,n_selected_observations,service_samples_seconds
activity,,,,,,,,,,
Book Vehicles,1526.067114,877.243889,204.0,3633.0,positive_sojourn,minimum,1330,594,149,"[204.0, 302.0, 319.0, 324.0, 334.0, 338.0, 353..."
Bring to Loading Bay,119.308316,28.836573,60.0,166.0,positive_sojourn,minimum,3934,1969,493,"[60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 66...."
Collect Goods,118.857901,49.580115,30.0,212.0,forward,minimum,10553,10553,2639,"[30.0, 31.0, 31.0, 31.0, 32.0, 32.0, 32.0, 32...."
Create Transport Document,1735.885906,1084.629634,302.0,3890.0,forward,minimum,594,594,149,"[302.0, 319.0, 324.0, 334.0, 338.0, 353.0, 353..."
Depart,40515.687500,29300.884261,51.0,69275.0,positive_sojourn,minimum,181,127,32,"[51.0, 769.0, 3878.0, 4996.0, 6520.0, 6769.0, ..."
Drive to Terminal,118.774874,54.462468,30.0,230.0,forward,minimum,3977,1989,995,"[30.0, 31.0, 31.0, 31.0, 32.0, 32.0, 32.0, 32...."
Load Truck,83.517241,28.029439,30.0,134.0,positive_sojourn,minimum,21096,10553,2639,"[30.0, 30.0, 30.0, 30.0, 31.0, 31.0, 31.0, 31...."
Load to Vehicle,88.896861,31.227835,30.0,146.0,positive_sojourn,minimum,5877,1966,446,"[30.0, 30.0, 31.0, 31.0, 32.0, 32.0, 33.0, 33...."
Order Empty Containers,4690.369128,1927.540866,825.0,7393.0,positive_sojourn,minimum,2569,593,149,"[825.0, 838.0, 838.0, 911.0, 942.0, 980.0, 101..."



simulated: 13 activities — durations in seconds


,service_mean,service_std,service_min,service_max,estimation_source,estimation_window,n_forward_observations,n_sojourn_observations,n_selected_observations,service_samples_seconds
activity,,,,,,,,,,
Book Vehicles,954.716175,11.915108,942.801067,966.631282,positive_sojourn,minimum,11,6,2,"[942.801067, 966.631282]"
Bring to Loading Bay,96.106922,11.687115,75.271980,109.072003,positive_sojourn,minimum,16,19,5,"[75.27198, 92.410141, 101.577694, 102.202791, ..."
Collect Goods,16872.406880,16698.176021,63.538804,53957.747683,forward,minimum,94,105,24,"[63.538804, 3163.206911, 4162.126611, 5365.488..."
Create Transport Document,1127.346011,160.714729,966.631282,1288.060740,forward,minimum,6,6,2,"[966.631282, 1288.06074]"
Drive to Terminal,27.122556,5.477617,19.487986,36.276107,forward,minimum,20,20,5,"[19.487986, 24.428997, 27.676746, 27.742942, 3..."
Load Truck,58.351494,8.804914,40.952520,70.105593,positive_sojourn,minimum,184,94,24,"[40.95252, 45.319791, 45.711961, 46.76593, 46...."
Load to Vehicle,52.870549,7.548077,34.742050,57.887726,positive_sojourn,minimum,60,25,7,"[34.74205, 53.203498, 54.389383, 56.346241, 56..."
Order Empty Containers,161.626925,102.821674,70.750337,355.346087,forward,minimum,31,6,8,"[70.750337, 76.92396, 89.171904, 89.171904, 10..."
Pick Up Empty Container,69.743417,23.216040,31.183925,96.273290,forward,minimum,20,27,5,"[31.183925, 64.78331, 65.289807, 91.186752, 96..."


WMAPE and Waserstein-1 between both logs:

WMAPE:

In [34]:
# ===== WMAPE on discovered service-time estimates ============================
# Requires SERVICE_TIMES from the new service-time discovery cell.
# Compares the already-discovered parameters; no pooling or rediscovery.

import numpy as np
import pandas as pd
from IPython.display import display

if "SERVICE_TIMES" not in globals():
    raise NameError(
        "SERVICE_TIMES is missing — run service-time discovery first."
    )

A = SERVICE_TIMES["input"]
F = SERVICE_TIMES["simulated"]

shared = sorted(set(A) & set(F))
only_in_input = sorted(set(A) - set(F))
only_in_simulated = sorted(set(F) - set(A))

if not shared:
    raise ValueError("No shared activities with service-time estimates.")

service = pd.DataFrame([
    {
        "activity": activity,
        "mean_h_input": A[activity]["service_mean"] / 3600.0,
        "mean_h_simulated": F[activity]["service_mean"] / 3600.0,
        "std_h_input": A[activity]["service_std"] / 3600.0,
        "std_h_simulated": F[activity]["service_std"] / 3600.0,
        "source_input": A[activity]["estimation_source"],
        "source_simulated": F[activity]["estimation_source"],
        "n_selected_input": A[activity]["n_selected_observations"],
        "n_selected_simulated": F[activity]["n_selected_observations"],
    }
    for activity in shared
])


def calculate_wmape(actual, simulated):
    actual = np.asarray(actual, dtype=float)
    simulated = np.asarray(simulated, dtype=float)

    if not (
        np.all(np.isfinite(actual))
        and np.all(np.isfinite(simulated))
    ):
        raise ValueError("WMAPE requires finite timing estimates.")

    denominator = np.abs(actual).sum()

    if denominator == 0:
        return np.nan

    return float(np.abs(actual - simulated).sum() / denominator)


wmape_mean = calculate_wmape(
    service["mean_h_input"],
    service["mean_h_simulated"],
)

wmape_std = calculate_wmape(
    service["std_h_input"],
    service["std_h_simulated"],
)


def format_wmape(value):
    return f"{value * 100:.2f}%" if np.isfinite(value) else "undefined"


print("WMAPE on discovered service-time estimates")
print(f"Activities compared: {len(shared)} of {len(set(A) | set(F))}")
print(f"WMAPE service mean: {format_wmape(wmape_mean)}")
print(f"WMAPE service std:  {format_wmape(wmape_std)}")
print(f"Only in input: {only_in_input}")
print(f"Only in simulated: {only_in_simulated}")

display(service.round(4))

WMAPE on discovered service-time estimates
Activities compared: 13 of 14
WMAPE service mean: 118.68%
WMAPE service std:  145.16%
Only in input: ['Depart']
Only in simulated: []


,activity,mean_h_input,mean_h_simulated,std_h_input,std_h_simulated,source_input,source_simulated,n_selected_input,n_selected_simulated
0,Book Vehicles,0.4239,0.2652,0.2437,0.0033,positive_sojourn,positive_sojourn,149,2
1,Bring to Loading Bay,0.0331,0.0267,0.0080,0.0032,positive_sojourn,positive_sojourn,493,5
2,Collect Goods,0.0330,4.6868,0.0138,4.6384,forward,forward,2639,24
3,Create Transport Document,0.4822,0.3132,0.3013,0.0446,forward,forward,149,2
4,Drive to Terminal,0.0330,0.0075,0.0151,0.0015,forward,forward,995,5
5,Load Truck,0.0232,0.0162,0.0078,0.0024,positive_sojourn,positive_sojourn,2639,24
6,Load to Vehicle,0.0247,0.0147,0.0087,0.0021,positive_sojourn,positive_sojourn,446,7
7,Order Empty Containers,1.3029,0.0449,0.5354,0.0286,positive_sojourn,forward,149,8
8,Pick Up Empty Container,0.6692,0.0194,0.3236,0.0064,forward,forward,499,5
9,Place in Stock,0.0183,0.0138,0.0052,0.0006,positive_sojourn,positive_sojourn,445,4


Wasserstein-1

In [36]:
# ===== Wasserstein-1 on selected service-time proxy samples ===================
# Requires SERVICE_TIMES containing service_samples_seconds for both logs.
#
# Uses ALL observations already retained by the discovery window.
# Does not reapply minimum/p25/p50 selection.
# Anchors are excluded because they supply parameters, not empirical samples.

import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance
from IPython.display import display

if "SERVICE_TIMES" not in globals():
    raise NameError(
        "SERVICE_TIMES is missing — run service-time discovery first."
    )

# W1 is defined for any nonempty sample.
# Increase this threshold if you want to exclude small selected samples.
MIN_SAMPLES = 1


def load_selected_samples(results, role):
    samples = {}

    for activity, stats in results.items():
        if "service_samples_seconds" not in stats:
            raise KeyError(
                f"{role}, {activity!r}: selected samples are missing. "
                "Update and rerun the service-time discovery cell."
            )

        values = np.asarray(
            stats["service_samples_seconds"], dtype=float
        )

        if (
            values.ndim != 1
            or not np.all(np.isfinite(values))
            or np.any(values < 0)
        ):
            raise ValueError(
                f"{role}, {activity!r}: samples must be a "
                "one-dimensional collection of finite, nonnegative durations."
            )

        samples[activity] = values / 3600.0  # Convert seconds to hours.

    return samples


reference = SERVICE_TIMES["input"]
simulated = SERVICE_TIMES["simulated"]

samples_input = load_selected_samples(reference, "input")
samples_simulated = load_selected_samples(simulated, "simulated")

all_activities = sorted(set(reference) | set(simulated))
rows = []
excluded = []

for activity in all_activities:
    if activity not in reference:
        excluded.append({
            "activity": activity,
            "reason": "Missing from input discovery results",
        })
        continue

    if activity not in simulated:
        excluded.append({
            "activity": activity,
            "reason": "Missing from simulated discovery results",
        })
        continue

    actual = reference[activity]
    generated = simulated[activity]

    if (
        actual["estimation_source"] == "anchor"
        or generated["estimation_source"] == "anchor"
    ):
        excluded.append({
            "activity": activity,
            "reason": "Anchor parameters used in one or both logs",
        })
        continue

    xa = samples_input[activity]
    xb = samples_simulated[activity]

    if len(xa) < MIN_SAMPLES or len(xb) < MIN_SAMPLES:
        excluded.append({
            "activity": activity,
            "reason": (
                f"Insufficient selected samples: "
                f"input={len(xa)}, simulated={len(xb)}"
            ),
        })
        continue

    if actual["estimation_window"] != generated["estimation_window"]:
        raise ValueError(
            f"{activity!r}: discovery windows differ between logs. "
            "Rerun discovery using the same mode for both logs."
        )

    distance = float(wasserstein_distance(xa, xb))
    mean_input = float(xa.mean())
    mean_simulated = float(xb.mean())

    rows.append({
        "activity": activity,
        "window": actual["estimation_window"],
        "source_input": actual["estimation_source"],
        "source_simulated": generated["estimation_source"],
        "n_input": len(xa),
        "n_simulated": len(xb),
        "mean_h_input": mean_input,
        "mean_h_simulated": mean_simulated,
        "W1_hours": distance,
        "W1_norm": (
            distance / mean_input if mean_input > 0 else np.nan
        ),
        "mean_gap_h": abs(mean_input - mean_simulated),
    })


columns = [
    "activity", "window", "source_input", "source_simulated",
    "n_input", "n_simulated",
    "mean_h_input", "mean_h_simulated",
    "W1_hours", "W1_norm", "mean_gap_h",
]

wf = pd.DataFrame(rows, columns=columns)
w1_excluded = pd.DataFrame(excluded, columns=["activity", "reason"])


def weighted_w1(frame):
    """Average per-activity W1, weighted by reference selected-sample counts."""
    if frame.empty:
        return np.nan

    return float(np.average(
        frame["W1_hours"].to_numpy(dtype=float),
        weights=frame["n_input"].to_numpy(dtype=float),
    ))


def selected_sample_coverage(samples, scored_activities, results):
    """Share of available non-anchor selected observations included in scoring."""
    eligible = {
        activity: values
        for activity, values in samples.items()
        if results[activity]["estimation_source"] != "anchor"
    }

    total = sum(len(values) for values in eligible.values())
    scored = sum(
        len(values)
        for activity, values in eligible.items()
        if activity in scored_activities
    )

    return scored / total if total > 0 else np.nan


scored_activities = set(wf["activity"])

w1_result = {
    "activities_scored": len(wf),
    "activities_total": len(all_activities),
    "W1_weighted_hours": weighted_w1(wf),
    "W1_unweighted_hours": (
        float(wf["W1_hours"].mean()) if not wf.empty else np.nan
    ),
    "W1_median_hours": (
        float(wf["W1_hours"].median()) if not wf.empty else np.nan
    ),
    "input_selected_sample_coverage": selected_sample_coverage(
        samples_input, scored_activities, reference
    ),
    "simulated_selected_sample_coverage": selected_sample_coverage(
        samples_simulated, scored_activities, simulated
    ),
}

print("Wasserstein-1 on selected service-time proxy distributions")

for name, value in w1_result.items():
    if name.startswith("activities_"):
        print(f"{name}: {value}")
    elif not np.isfinite(value):
        print(f"{name}: undefined")
    elif name.endswith("_hours"):
        print(f"{name}: {value:.4f} h")
    else:
        print(f"{name}: {value:.2%}")

if not wf.empty:
    print("\nPer activity, largest distribution differences first:")
    display(
        wf.sort_values("W1_hours", ascending=False)
        .reset_index(drop=True)
        .round(4)
    )
else:
    print("\nNo activities qualified for comparison.")

if not w1_excluded.empty:
    print("\nExcluded activities:")
    display(w1_excluded)

Wasserstein-1 on selected service-time proxy distributions
activities_scored: 13
activities_total: 14
W1_weighted_hours: 1.5319 h
W1_unweighted_hours: 1.5887 h
W1_median_hours: 0.0255 h
input_selected_sample_coverage: 99.67%
simulated_selected_sample_coverage: 100.00%

Per activity, largest distribution differences first:


,activity,window,source_input,source_simulated,n_input,n_simulated,mean_h_input,mean_h_simulated,W1_hours,W1_norm,mean_gap_h
0,Register Customer Order,minimum,forward,forward,149,2,14.2104,0.6429,13.5675,0.9548,13.5675
1,Collect Goods,minimum,forward,forward,2639,24,0.0330,4.6868,4.6538,140.9544,4.6538
2,Order Empty Containers,minimum,positive_sojourn,forward,149,8,1.3029,0.0449,1.2580,0.9655,1.2580
3,Pick Up Empty Container,minimum,forward,forward,499,5,0.6692,0.0194,0.6498,0.9710,0.6498
4,Create Transport Document,minimum,forward,forward,149,2,0.4822,0.3132,0.2289,0.4747,0.1690
5,Book Vehicles,minimum,positive_sojourn,positive_sojourn,149,2,0.4239,0.2652,0.2205,0.5201,0.1587
6,Drive to Terminal,minimum,forward,forward,995,5,0.0330,0.0075,0.0255,0.7716,0.0255
7,Reschedule Container,minimum,positive_sojourn,positive_sojourn,9,3,0.0282,0.0189,0.0105,0.3727,0.0093
8,Load to Vehicle,minimum,positive_sojourn,positive_sojourn,446,7,0.0247,0.0147,0.0101,0.4074,0.0100
9,Weigh,minimum,forward,positive_sojourn,995,5,0.0136,0.0073,0.0087,0.6449,0.0063



Excluded activities:


,activity,reason
0,Depart,Missing from simulated discovery results


NGD:

In [37]:

# ===== Relative NGD — object-level activity sequences ========================
# Requires INPUT_LOG and SIMULATED_LOG.
#
# Based on the padded n-gram construction of Chapela-Campa et al. (2023),
# "Can I Trust My Simulation Model?", Section 4.1.
#
# Adaptations:
#   1. One sequence per object, rather than per process case.
#   2. Total variation between separately normalized n-gram histograms:
#          0.5 * sum_g |count_input(g)/N_input - count_simulated(g)/N_simulated|
#
# The published count-based NGD instead divides the sum of absolute
# count differences by N_input + N_simulated.
#
# Equal timestamps: event ID provides a deterministic tie-break.
# This convention does not establish the true order of simultaneous events.

import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

NS = (2, 3)

# A shared non-string token cannot collide with a string activity name.
_NGD_PADDING = object()


def load_ngd_log(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    events = raw.get("events")
    objects = raw.get("objects")

    if not isinstance(events, list) or not isinstance(objects, list):
        raise ValueError("Expected OCEL 2.0 'events' and 'objects' lists.")

    object_types = {}

    for obj in objects:
        oid = obj["id"]
        otype = obj["type"]

        if not isinstance(oid, str) or not isinstance(otype, str):
            raise ValueError("Object IDs and types must be strings.")
        if oid in object_types:
            raise ValueError(f"Duplicate object ID: {oid!r}")

        object_types[oid] = otype

    # Validate timezone information rather than silently assuming local time.
    timestamps = []
    seen_event_ids = set()

    for event in events:
        eid = event["id"]
        activity = event["type"]

        if not isinstance(eid, str) or not isinstance(activity, str):
            raise ValueError("Event IDs and activity names must be strings.")
        if eid in seen_event_ids:
            raise ValueError(f"Duplicate event ID: {eid!r}")
        seen_event_ids.add(eid)

        timestamp = pd.Timestamp(event.get("time"))

        if pd.isna(timestamp) or timestamp.tzinfo is None:
            raise ValueError(
                f"Event {eid!r} needs a valid timestamp with a timezone."
            )

        timestamps.append(timestamp.tz_convert("UTC"))

    timelines = defaultdict(list)
    unattached_events = 0
    repeated_attachments = 0

    for event, timestamp in zip(events, timestamps):
        object_ids = [
            relation["objectId"]
            for relation in event.get("relationships") or []
        ]

        # One occurrence of an event per object, regardless of qualifier.
        unique_ids = set(object_ids)
        repeated_attachments += len(object_ids) - len(unique_ids)

        if not unique_ids:
            unattached_events += 1

        for oid in unique_ids:
            if oid not in object_types:
                raise ValueError(
                    f"Event {event['id']!r} references unknown object {oid!r}."
                )

            timelines[oid].append(
                (timestamp, event["id"], event["type"])
            )

    sequences = defaultdict(list)
    tied_timelines = 0

    for oid, timeline in timelines.items():
        timeline.sort(key=lambda item: (item[0], item[1]))

        if any(
            left[0] == right[0]
            for left, right in zip(timeline, timeline[1:])
        ):
            tied_timelines += 1

        sequences[object_types[oid]].append(
            [activity for _, _, activity in timeline]
        )

    return {
        "sequences": dict(sequences),
        "declared_types": set(object_types.values()),
        "unattached_events": unattached_events,
        "unused_objects": len(object_types) - len(timelines),
        "repeated_attachments_removed": repeated_attachments,
        "timelines_with_timestamp_ties": tied_timelines,
    }


def ngd_histogram(log, object_type=None, n=2):
    if not isinstance(n, int) or n < 1:
        raise ValueError("n must be a positive integer.")

    if object_type is None:
        sequences = (
            sequence
            for type_sequences in log["sequences"].values()
            for sequence in type_sequences
        )
    else:
        sequences = iter(log["sequences"].get(object_type, []))

    counts = Counter()
    n_traces = 0

    for sequence in sequences:
        padded = (
            [_NGD_PADDING] * (n - 1)
            + sequence
            + [_NGD_PADDING] * (n - 1)
        )

        counts.update(
            tuple(padded[i:i + n])
            for i in range(len(padded) - n + 1)
        )
        n_traces += 1

    return counts, n_traces


def relative_ngd(reference, simulated, object_type=None, n=2):
    c_input, traces_input = ngd_histogram(reference, object_type, n)
    c_simulated, traces_simulated = ngd_histogram(simulated, object_type, n)

    total_input = sum(c_input.values())
    total_simulated = sum(c_simulated.values())

    if total_input == 0 and total_simulated == 0:
        distance = np.nan
        status = "No observed traces in either log"
    elif total_input == 0:
        distance = np.nan
        status = "No observed traces in input"
    elif total_simulated == 0:
        distance = np.nan
        status = "No observed traces in simulation"
    else:
        grams = set(c_input) | set(c_simulated)

        distance = 0.5 * sum(
            abs(
                c_input.get(g, 0) / total_input
                - c_simulated.get(g, 0) / total_simulated
            )
            for g in grams
        )
        status = "Compared"

    return {
        "scope": "pooled" if object_type is None else "object_type",
        "object_type": object_type,
        "n": n,
        "traces_input": traces_input,
        "traces_simulated": traces_simulated,
        "ngrams_input": total_input,
        "ngrams_simulated": total_simulated,
        "NGD_relative": distance,
        "status": status,
    }


# Always reload: changing the selected files must update the calculation.
log_in_ngd = load_ngd_log(INPUT_LOG)
log_sim_ngd = load_ngd_log(SIMULATED_LOG)

types = sorted(
    log_in_ngd["declared_types"] | log_sim_ngd["declared_types"]
)

rows = []

for n in NS:
    for object_type in types:
        rows.append(
            relative_ngd(log_in_ngd, log_sim_ngd, object_type, n)
        )

    rows.append(relative_ngd(log_in_ngd, log_sim_ngd, None, n))

ngd_table = pd.DataFrame(rows)

pooled = ngd_table.loc[ngd_table["scope"] == "pooled"]

ngd_result = {
    f"NGD_{int(row['n'])}gram_relative": float(row["NGD_relative"])
    for _, row in pooled.iterrows()
}

print("Relative NGD — normalized object-level n-gram distributions")
print("Range: 0 = identical frequencies; 1 = no shared n-grams.")

for name, value in ngd_result.items():
    print(f"{name}: {value:.4f}" if np.isfinite(value)
          else f"{name}: undefined")

print(
    "\nTypes declared only in input:",
    sorted(log_in_ngd["declared_types"] - log_sim_ngd["declared_types"]),
)
print(
    "Types declared only in simulation:",
    sorted(log_sim_ngd["declared_types"] - log_in_ngd["declared_types"]),
)

for role, log in (("input", log_in_ngd), ("simulated", log_sim_ngd)):
    print(
        f"\n{role}: "
        f"{log['timelines_with_timestamp_ties']} object timelines with time ties; "
        f"{log['repeated_attachments_removed']} repeated attachments removed; "
        f"{log['unattached_events']} unattached events excluded; "
        f"{log['unused_objects']} objects without event traces."
    )

print("\nPer-type and pooled results:")
display(ngd_table.round({"NGD_relative": 4}))

Relative NGD — normalized object-level n-gram distributions
Range: 0 = identical frequencies; 1 = no shared n-grams.
NGD_2gram_relative: 0.2780
NGD_3gram_relative: 0.3569

Types declared only in input: []
Types declared only in simulation: []

input: 154 object timelines with time ties; 0 repeated attachments removed; 0 unattached events excluded; 0 objects without event traces.

simulated: 0 object timelines with time ties; 0 repeated attachments removed; 0 unattached events excluded; 0 objects without event traces.

Per-type and pooled results:


,scope,object_type,n,traces_input,traces_simulated,ngrams_input,ngrams_simulated,NGD_relative,status
0,object_type,Container,2,1999,27,28265,287,0.3102,Compared
1,object_type,Customer Order,2,600,6,1794,18,0.0045,Compared
2,object_type,Forklift,2,3,19,7741,99,0.7971,Compared
3,object_type,Handling Unit,2,10553,105,31659,304,0.0482,Compared
4,object_type,Transport Document,2,594,6,3165,36,0.5997,Compared
5,object_type,Truck,2,6,20,12548,134,0.2979,Compared
6,object_type,Vehicle,2,127,2,2982,45,0.4533,Compared
7,pooled,NaN,2,13882,185,88154,923,0.2780,Compared
8,object_type,Container,3,1999,27,30264,314,0.4403,Compared
9,object_type,Customer Order,3,600,6,2394,24,0.0056,Compared


Simulated Log vs Model:

OC-Declare Confidence Score:

In [41]:
# ===== OC-Declare confidence — Definitions 8 & 9 ==============================
# Requires: INPUT_LOG, SIMULATED_LOG, OCDECLARE_MODEL
#
# Supported: EF / EP with direct object-type Each / All / Any involvement.
# Canonical arc orientation: "from" activates; "to" is the counted activity.
# counts = [minimum, maximum], where null means no upper bound.

import json
import itertools
import numpy as np
import pandas as pd
from collections import defaultdict
from IPython.display import display


# ---- Model loading and validation ------------------------------------------

def load_confidence_model(path):
    with open(path, encoding="utf-8") as f:
        model = json.load(f)

    if model.get("constraint_orientation") != "arc":
        raise ValueError(
            "Expected constraint_orientation='arc'. "
            "Other orientations require explicit conversion."
        )

    constraints = []

    for i, c in enumerate(model["constraints"], start=1):
        ar = str(c["arc_type"]).upper()
        if ar not in {"EF", "EP"}:
            raise NotImplementedError(
                f"Constraint {i}: unsupported arc type {ar!r}."
            )

        bounds = c.get("counts")
        if not isinstance(bounds, list) or len(bounds) != 2:
            raise ValueError(
                f"Constraint {i}: expected counts=[minimum, maximum]."
            )

        lower, upper = bounds
        if type(lower) is not int or lower < 0:
            raise ValueError(
                f"Constraint {i}: minimum must be a nonnegative integer."
            )
        if upper is not None and (
            type(upper) is not int or upper < lower
        ):
            raise ValueError(
                f"Constraint {i}: maximum must be null "
                "or an integer >= minimum."
            )

        inv = c.get("involvement_per_label")
        if not isinstance(inv, dict):
            raise ValueError(
                f"Constraint {i}: involvement_per_label must be a dictionary."
            )

        inv = {label: str(mode).lower() for label, mode in inv.items()}
        for label, mode in inv.items():
            if mode not in {"each", "all", "any"}:
                raise ValueError(
                    f"Constraint {i}: unknown involvement mode {mode!r}."
                )
            if ">" in label or "<" in label:
                raise NotImplementedError(
                    f"Constraint {i}: indirect object label {label!r} "
                    "requires object-to-object relationship handling."
                )

        constraints.append({
            **c,
            "constraint_id": i,
            "arc_type": ar,
            "source_activity": c["from"],
            "target_activity": c["to"],
            "nmin": lower,
            "nmax": np.inf if upper is None else upper,
            "involvement_per_label": inv,
        })

    return {**model, "constraints": constraints}


# ---- OCEL index -------------------------------------------------------------

def build_index(path):
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    objects = raw["objects"]
    events = raw["events"]

    otype = {o["id"]: o["type"] for o in objects}
    if len(otype) != len(objects):
        raise ValueError(f"{path}: duplicate object IDs.")

    if len({e["id"] for e in events}) != len(events):
        raise ValueError(f"{path}: duplicate event IDs.")

    timestamps = pd.to_datetime(
        [e["time"] for e in events],
        utc=True,
        format="ISO8601",
        errors="raise",
    )
    if timestamps.isna().any():
        raise ValueError(f"{path}: missing event timestamps.")

    # The paper assumes unique timestamps. Do not invent an order for ties.
    n_tied_events = int(timestamps.duplicated(keep=False).sum())

    if n_tied_events:
        print(
            f"Note: {path}\n"
            f"  {n_tied_events} events share a timestamp with another event.\n"
            "  Equal-time events are treated as simultaneous and cannot "
            "fulfil an EF/EP requirement for each other."
        )

    ev_objs = defaultdict(lambda: defaultdict(set))
    obj_events = defaultdict(set)
    ev_time = {}
    act_events = defaultdict(set)

    for event, timestamp in zip(events, timestamps):
        eid = event["id"]
        ev_time[eid] = timestamp
        act_events[event["type"]].add(eid)

        for relation in event.get("relationships") or []:
            oid = relation["objectId"]
            if oid not in otype:
                raise ValueError(
                    f"{path}: event {eid!r} references unknown object {oid!r}."
                )

            ev_objs[eid][otype[oid]].add(oid)
            obj_events[oid].add(eid)

    return {
        "ev_objs": ev_objs,
        "obj_events": obj_events,
        "ev_time": ev_time,
        "act_events": act_events,
        "n_events": len(events),
    }


# ---- Definition 8: satisfaction of one activation ----------------------------

def involvement(c):
    inv = c["involvement_per_label"]
    return (
        [label for label, mode in inv.items() if mode == "each"],
        [label for label, mode in inv.items() if mode == "all"],
        [label for label, mode in inv.items() if mode == "any"],
    )


def matching_events(idx, eid, c, each_combo, all_objects, any_types):
    """Return matching target events for one Cartesian Each binding."""
    target = c["target_activity"]
    candidates = set(idx["act_events"].get(target, set()))

    # Every selected Each object and every All object must be present.
    required = set(each_combo) | all_objects
    for oid in required:
        candidates.intersection_update(idx["obj_events"].get(oid, set()))

    # Each Any type requires at least one object shared with the activation.
    for object_type in any_types:
        source_objects = idx["ev_objs"][eid].get(object_type, set())
        candidates = {
            other for other in candidates
            if idx["ev_objs"][other].get(object_type, set()) & source_objects
        }

    timestamp = idx["ev_time"][eid]
    if c["arc_type"] == "EF":
        return {
            other for other in candidates
            if idx["ev_time"][other] > timestamp
        }

    # EP: source still activates, but matching targets must be earlier.
    return {
        other for other in candidates
        if idx["ev_time"][other] < timestamp
    }


def satisfies(idx, eid, c):
    """Check nmin <= matching count <= nmax for EVERY Each binding."""
    each, alls, anys = involvement(c)
    objects = idx["ev_objs"][eid]

    all_objects = set()
    for object_type in alls:
        all_objects.update(objects.get(object_type, set()))

    domains = [
        sorted(objects.get(object_type, set()))
        for object_type in each
    ]

    counts = []

    # With no Each labels, product() yields one empty binding.
    # With an empty Each domain, it yields no bindings: vacuous satisfaction.
    for combo in itertools.product(*domains):
        counts.append(len(
            matching_events(idx, eid, c, combo, all_objects, anys)
        ))

    ok = all(c["nmin"] <= count <= c["nmax"] for count in counts)
    return ok, counts


# ---- Definition 9: per-constraint and global confidence ----------------------

REPORT_COLUMNS = [
    "constraint_id", "arc", "activates_on", "counts_activity",
    "each", "all", "any", "nmin", "nmax",
    "n_activations", "n_satisfied", "n_violating",
    "n_vacuous_activations", "confidence", "mean_binding_count",
    "model_support",
]


def evaluate_confidence(idx, model):
    rows = []
    violating_events = set()

    for c in model["constraints"]:
        each, alls, anys = involvement(c)
        activations = idx["act_events"].get(c["source_activity"], set())

        satisfied = 0
        vacuous = 0
        binding_counts = []

        for eid in activations:
            ok, counts = satisfies(idx, eid, c)
            satisfied += int(ok)
            vacuous += int(len(counts) == 0)
            binding_counts.extend(counts)

            if not ok:
                violating_events.add(eid)

        n = len(activations)
        rows.append({
            "constraint_id": c["constraint_id"],
            "arc": c["arc_type"],
            "activates_on": c["source_activity"],
            "counts_activity": c["target_activity"],
            "each": ", ".join(each),
            "all": ", ".join(alls),
            "any": ", ".join(anys),
            "nmin": c["nmin"],
            "nmax": c["nmax"],
            "n_activations": n,
            "n_satisfied": satisfied,
            "n_violating": n - satisfied,
            "n_vacuous_activations": vacuous,
            # No activations means undefined confidence, not 100%.
            "confidence": satisfied / n if n else np.nan,
            "mean_binding_count": (
                float(np.mean(binding_counts)) if binding_counts else np.nan
            ),
            "model_support": c.get("support", np.nan),
        })

    table = pd.DataFrame(rows, columns=REPORT_COLUMNS)
    n_events = idx["n_events"]
    n_bad = len(violating_events)

    summary = {
        "n_events": n_events,
        "n_satisfying_events": n_events - n_bad,
        "n_violating_events": n_bad,
        "n_constraints": len(table),
        "n_inactive_constraints": int((table["n_activations"] == 0).sum()),
        "global_confidence": (
            (n_events - n_bad) / n_events if n_events else np.nan
        ),
        # Additional summary; this is NOT Definition 9's global confidence.
        "mean_constraint_confidence": (
            float(table["confidence"].mean())
            if table["confidence"].notna().any() else np.nan
        ),
    }

    return table, summary


# ---- Evaluate the SAME model against both logs ------------------------------

model = load_confidence_model(OCDECLARE_MODEL)

idx_in = build_index(INPUT_LOG)
idx_sim = build_index(SIMULATED_LOG)

conf_in, summary_in = evaluate_confidence(idx_in, model)
conf_sim, summary_sim = evaluate_confidence(idx_sim, model)

confidence_summary = pd.DataFrame(
    [summary_in, summary_sim],
    index=pd.Index(["input", "simulated"], name="log"),
)

# Retained as numeric values for later cells.
confidence_result = {
    "input_gconf": summary_in["global_confidence"],
    "simulated_gconf": summary_sim["global_confidence"],
    "input_mean_confidence": summary_in["mean_constraint_confidence"],
    "simulated_mean_confidence": summary_sim["mean_constraint_confidence"],
}

print("OC-Declare confidence: 0 = none, 1 = fully conforming")
print("Global confidence: share of events satisfying all applicable constraints.")
print("Mean constraint confidence: additional average over activated constraints.")
print("No-activation constraints have undefined per-constraint confidence.\n")

display(confidence_summary.round(6))

print("\nPer-constraint confidence — input:")
display(conf_in.round(6))

print("\nPer-constraint confidence — simulated:")
display(conf_sim.round(6))


Note: /Users/luisschwarz/Documents/Studies/IT/ProjectTest/ProjectTest/Backend/src/Simulation/IO/input/eventlog/container_logistics.json
  13822 events share a timestamp with another event.
  Equal-time events are treated as simultaneous and cannot fulfil an EF/EP requirement for each other.
OC-Declare confidence: 0 = none, 1 = fully conforming
Global confidence: share of events satisfying all applicable constraints.
Mean constraint confidence: additional average over activated constraints.
No-activation constraints have undefined per-constraint confidence.



,n_events,n_satisfying_events,n_violating_events,n_constraints,n_inactive_constraints,global_confidence,mean_constraint_confidence
log,,,,,,,
input,35372,34747,625,35,0,0.982331,0.986143
simulated,362,212,150,35,4,0.585635,0.706256



Per-constraint confidence — input:


,constraint_id,arc,activates_on,counts_activity,each,all,any,nmin,nmax,n_activations,n_satisfied,n_violating,n_vacuous_activations,confidence,mean_binding_count,model_support
0,1,EP,Book Vehicles,Create Transport Document,Transport Document,,,1,inf,594,594,0,0,1.000000,1.000000,1.0000
1,2,EF,Book Vehicles,Depart,"Transport Document, Vehicle",,,1,inf,594,581,13,20,0.978114,0.982361,0.9774
2,3,EF,Book Vehicles,Order Empty Containers,Transport Document,,,1,inf,594,593,1,0,0.998316,0.998316,0.9983
3,4,EF,Bring to Loading Bay,Load to Vehicle,"Container, Forklift",,,1,inf,1969,1943,26,0,0.986795,0.986795,0.9868
4,5,EP,Bring to Loading Bay,Place in Stock,Container,,,1,inf,1969,1794,175,0,0.911122,0.911122,0.9111
5,6,EF,Collect Goods,Load Truck,Handling Unit,,,1,inf,10553,10553,0,0,1.000000,1.000000,1.0000
6,7,EF,Create Transport Document,Book Vehicles,Transport Document,,,1,inf,594,594,0,0,1.000000,1.000000,1.0000
7,8,EP,Create Transport Document,Register Customer Order,Customer Order,,,1,inf,594,594,0,0,1.000000,1.000000,1.0000
8,9,EP,Depart,Book Vehicles,"Transport Document, Vehicle",,,1,inf,127,112,15,0,0.881890,0.960212,0.8819
9,10,EP,Depart,Load to Vehicle,"Container, Vehicle",,,1,inf,127,127,0,0,1.000000,1.000000,1.0000



Per-constraint confidence — simulated:


,constraint_id,arc,activates_on,counts_activity,each,all,any,nmin,nmax,n_activations,n_satisfied,n_violating,n_vacuous_activations,confidence,mean_binding_count,model_support
0,1,EP,Book Vehicles,Create Transport Document,Transport Document,,,1,inf,6,6,0,0,1.000000,1.000000,1.0000
1,2,EF,Book Vehicles,Depart,"Transport Document, Vehicle",,,1,inf,6,0,6,0,0.000000,0.000000,0.9774
2,3,EF,Book Vehicles,Order Empty Containers,Transport Document,,,1,inf,6,4,2,0,0.666667,0.666667,0.9983
3,4,EF,Bring to Loading Bay,Load to Vehicle,"Container, Forklift",,,1,inf,19,0,19,0,0.000000,0.000000,0.9868
4,5,EP,Bring to Loading Bay,Place in Stock,Container,,,1,inf,19,16,3,0,0.842105,0.842105,0.9111
5,6,EF,Collect Goods,Load Truck,Handling Unit,,,1,inf,105,94,11,0,0.895238,0.895238,1.0000
6,7,EF,Create Transport Document,Book Vehicles,Transport Document,,,1,inf,6,6,0,0,1.000000,1.000000,1.0000
7,8,EP,Create Transport Document,Register Customer Order,Customer Order,,,1,inf,6,6,0,0,1.000000,1.000000,1.0000
8,9,EP,Depart,Book Vehicles,"Transport Document, Vehicle",,,1,inf,0,0,0,0,NaN,NaN,0.8819
9,10,EP,Depart,Load to Vehicle,"Container, Vehicle",,,1,inf,0,0,0,0,NaN,NaN,1.0000


Constraint Violations (object-centric, per-activation):

In [51]:
# ===== OC-Declare constraint violations — completed logs =====================
#
# Requires the helper functions from the corrected confidence cell.
# Reloads OCDECLARE_MODEL, INPUT_LOG, and SIMULATED_LOG.
#
# Activation violation rate:
#   failed event–constraint activations / all event–constraint activations
#
# Event violation rate:
#   unique events violating at least one constraint / all events
#   = 1 - global confidence
#
# Each failed activation counts once, regardless of how many
# object combinations fail. No RV-LTL repairability classification.
# Detailed tables are stored but not displayed.

import numpy as np
import pandas as pd


# ---- Check dependencies -----------------------------------------------------

_required = [
    "INPUT_LOG",
    "SIMULATED_LOG",
    "OCDECLARE_MODEL",
    "load_confidence_model",
    "build_index",
    "satisfies",
    "evaluate_confidence",
]

_missing = [name for name in _required if name not in globals()]
if _missing:
    raise NameError(
        "Run the corrected OC-Declare confidence cell first. "
        f"Missing: {', '.join(_missing)}"
    )


# ---- Violation evaluation ---------------------------------------------------

INCIDENT_COLUMNS = [
    "constraint_id",
    "arc",
    "activation_activity",
    "counted_activity",
    "event",
    "time",
    "nmin",
    "nmax",
    "n_evaluated_bindings",
    "n_failed_bindings",
    "minimum_observed_count",
    "maximum_observed_count",
]


def constraint_violation_report(idx, normalized_model):
    # Use the same satisfaction semantics as the confidence evaluator.
    per_constraint, confidence_summary = evaluate_confidence(
        idx, normalized_model
    )

    per_constraint = per_constraint.copy()
    denominator = per_constraint["n_activations"].replace(0, np.nan)
    per_constraint["violation_rate"] = (
        per_constraint["n_violating"] / denominator
    )

    incidents = []

    for c in normalized_model["constraints"]:
        activations = idx["act_events"].get(
            c["source_activity"], set()
        )

        for eid in sorted(
            activations,
            key=lambda event_id: (
                idx["ev_time"][event_id],
                str(event_id),
            ),
        ):
            ok, counts = satisfies(idx, eid, c)
            if ok:
                continue

            n_failed = sum(
                not (c["nmin"] <= count <= c["nmax"])
                for count in counts
            )

            # One row per failed event–constraint activation.
            incidents.append({
                "constraint_id": c["constraint_id"],
                "arc": c["arc_type"],
                "activation_activity": c["source_activity"],
                "counted_activity": c["target_activity"],
                "event": eid,
                "time": idx["ev_time"][eid],
                "nmin": c["nmin"],
                "nmax": c["nmax"],
                "n_evaluated_bindings": len(counts),
                "n_failed_bindings": n_failed,
                "minimum_observed_count": min(counts),
                "maximum_observed_count": max(counts),
            })

    incidents = pd.DataFrame(
        incidents,
        columns=INCIDENT_COLUMNS,
    )

    n_activations = int(per_constraint["n_activations"].sum())
    n_violations = int(per_constraint["n_violating"].sum())
    n_events = idx["n_events"]
    n_affected_events = int(incidents["event"].nunique())

    if len(incidents) != n_violations:
        raise RuntimeError(
            "Incident count differs from the confidence evaluator."
        )

    if n_affected_events != confidence_summary["n_violating_events"]:
        raise RuntimeError(
            "Violating-event count differs from the confidence evaluator."
        )

    summary = {
        "n_events": n_events,
        "n_constraints": len(per_constraint),
        "n_inactive_constraints": int(
            (per_constraint["n_activations"] == 0).sum()
        ),
        "n_activations": n_activations,
        "n_violating_activations": n_violations,
        "activation_violation_rate": (
            n_violations / n_activations
            if n_activations else np.nan
        ),
        "n_constraints_affected": int(
            (per_constraint["n_violating"] > 0).sum()
        ),
        "n_events_affected": n_affected_events,
        "event_violation_rate": (
            n_affected_events / n_events
            if n_events else np.nan
        ),
        "n_vacuous_activations": int(
            per_constraint["n_vacuous_activations"].sum()
        ),
    }

    return incidents, per_constraint, summary


# ---- Reload the normalized model and selected logs --------------------------
# Separate variable names avoid overwriting the confidence cell's model/indexes.

violation_model = load_confidence_model(OCDECLARE_MODEL)
violation_idx_in = build_index(INPUT_LOG)
violation_idx_sim = build_index(SIMULATED_LOG)


# ---- Evaluate both logs against the same model ------------------------------

viol_input, violations_by_constraint_input, violation_summary_input = (
    constraint_violation_report(violation_idx_in, violation_model)
)

viol_sim, violations_by_constraint_sim, violation_summary_sim = (
    constraint_violation_report(violation_idx_sim, violation_model)
)

# Store tables without displaying them.
violation_summary = pd.DataFrame(
    [violation_summary_input, violation_summary_sim],
    index=pd.Index(["input", "simulated"], name="log"),
)

# Numeric rates remain fractions in [0, 1].
violation_result = {
    "input_activation_violation_rate":
        violation_summary_input["activation_violation_rate"],
    "simulated_activation_violation_rate":
        violation_summary_sim["activation_violation_rate"],
    "input_event_violation_rate":
        violation_summary_input["event_violation_rate"],
    "simulated_event_violation_rate":
        violation_summary_sim["event_violation_rate"],
}


# ---- Print summaries only ---------------------------------------------------

def _violation_percent(value):
    return "undefined" if pd.isna(value) else f"{value:.2%}"


print("OC-Declare constraint violations — completed logs")
print("Lower violation rates are better.")
print("Event violation rate equals 1 - global confidence.")

for role, summary in [
    ("INPUT", violation_summary_input),
    ("SIMULATED", violation_summary_sim),
]:
    print(f"\n{role}")
    print(
        "  Activation violation rate : "
        f"{_violation_percent(summary['activation_violation_rate'])}"
        f"  ({summary['n_violating_activations']}"
        f"/{summary['n_activations']} activations)"
    )
    print(
        "  Event violation rate      : "
        f"{_violation_percent(summary['event_violation_rate'])}"
        f"  ({summary['n_events_affected']}"
        f"/{summary['n_events']} events)"
    )
    print(
        "  Constraints with violations: "
        f"{summary['n_constraints_affected']}"
        f"/{summary['n_constraints']}"
    )
    print(
        "  Constraints without activations: "
        f"{summary['n_inactive_constraints']}"
    )
    print(
        "  Vacuously satisfied activations: "
        f"{summary['n_vacuous_activations']}"
    )


OC-Declare constraint violations — completed logs
Lower violation rates are better.
Event violation rate equals 1 - global confidence.

INPUT
  Activation violation rate : 0.89%  (654/73659 activations)
  Event violation rate      : 1.77%  (625/35372 events)
  Constraints with violations: 14/35
  Constraints without activations: 0
  Vacuously satisfied activations: 47

SIMULATED
  Activation violation rate : 24.87%  (194/780 activations)
  Event violation rate      : 41.44%  (150/362 events)
  Constraints with violations: 16/35
  Constraints without activations: 4
  Vacuously satisfied activations: 0


Coverage of Elements:

In [52]:
# ===== OC-Declare model coverage — separate diagnostic scores ================
#
# Requires the corrected confidence cell's helper functions.
# Uses the same EF/EP, object-involvement, and timestamp semantics.
#
# No combined Overall score.
# Coverage measures describe what was observed, not necessarily conformance.

import json
import numpy as np
import pandas as pd


# ---- Dependencies -----------------------------------------------------------

_required = [
    "INPUT_LOG",
    "SIMULATED_LOG",
    "OCDECLARE_MODEL",
    "load_confidence_model",
    "build_index",
    "satisfies",
]

_missing = [name for name in _required if name not in globals()]
if _missing:
    raise NameError(
        "Run the corrected OC-Declare confidence cell first. "
        f"Missing: {', '.join(_missing)}"
    )


# ---- Helpers ----------------------------------------------------------------

def _coverage_fraction(numerator, denominator):
    return numerator / denominator if denominator else np.nan


def _coverage_mean(values):
    finite = [float(v) for v in values if pd.notna(v)]
    return float(np.mean(finite)) if finite else np.nan


def _coverage_index(path):
    # Reuse the corrected index and its input validation.
    idx = build_index(path)

    # The corrected confidence index does not retain all declared objects.
    # Include objects with no events when measuring object-type presence.
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    types_present = {obj["type"] for obj in raw["objects"]}
    return idx, types_present


# ---- Coverage calculation ---------------------------------------------------

COVERAGE_DETAIL_COLUMNS = [
    "constraint_id",
    "arc",
    "source_activity",
    "target_activity",
    "n_activations",
    "n_nonvacuous_activations",
    "source_activated",
    "nonvacuously_activated",
    "object_presence_coverage",
    "finite_boundary_coverage",
    "matching_event_coverage",
    "nmin",
    "nmax",
    "observed_counts",
]


def model_coverage(idx, types_present, normalized_model):
    model_activities = {
        activity["name"]
        for activity in normalized_model["activities"]
    }
    model_types = {
        object_type["name"]
        for object_type in normalized_model["object_types"]
    }
    log_activities = {
        activity
        for activity, events in idx["act_events"].items()
        if events
    }

    activity_coverage = _coverage_fraction(
        len(model_activities & log_activities),
        len(model_activities),
    )
    object_type_coverage = _coverage_fraction(
        len(model_types & types_present),
        len(model_types),
    )

    rows = []

    for c in normalized_model["constraints"]:
        events = idx["act_events"].get(
            c["source_activity"], set()
        )
        labels = list(c["involvement_per_label"])

        n_full_object_presence = 0
        n_nonvacuous = 0
        observed_counts = set()

        for eid in events:
            objects = idx["ev_objs"][eid]

            # Presence diagnostic only:
            # Each, All, and Any labels are all checked for object presence.
            # With no labels, the presence requirement holds trivially.
            if all(objects.get(label, set()) for label in labels):
                n_full_object_presence += 1

            # Reuse the corrected matching semantics.
            # counts has one entry per evaluated Cartesian Each binding.
            # Empty counts means a vacuous activation due to an empty
            # Each domain. With no Each labels, there is one binding.
            _, counts = satisfies(idx, eid, c)

            if counts:
                n_nonvacuous += 1
                observed_counts.update(counts)

        # Only finite boundaries are observable.
        # A set avoids counting an exact-count bound twice.
        boundaries = {c["nmin"]}
        if np.isfinite(c["nmax"]):
            boundaries.add(c["nmax"])

        boundary_coverage = (
            len(boundaries & observed_counts) / len(boundaries)
            if observed_counts else np.nan
        )

        # Matching-event coverage is interpreted only for constraints
        # requiring at least one matching event. Zero-minimum constraints
        # are excluded because observing a match need not be desirable.
        matching_coverage = (
            float(any(count > 0 for count in observed_counts))
            if c["nmin"] > 0 else np.nan
        )

        rows.append({
            "constraint_id": c["constraint_id"],
            "arc": c["arc_type"],
            "source_activity": c["source_activity"],
            "target_activity": c["target_activity"],
            "n_activations": len(events),
            "n_nonvacuous_activations": n_nonvacuous,
            "source_activated": bool(events),
            "nonvacuously_activated": n_nonvacuous > 0,
            "object_presence_coverage": _coverage_fraction(
                n_full_object_presence, len(events)
            ),
            "finite_boundary_coverage": boundary_coverage,
            "matching_event_coverage": matching_coverage,
            "nmin": c["nmin"],
            "nmax": c["nmax"],
            "observed_counts": sorted(observed_counts),
        })

    details = pd.DataFrame(rows, columns=COVERAGE_DETAIL_COLUMNS)

    scores = {
        # Presence among all declared model activities / object types.
        "Activity coverage": activity_coverage,
        "Object-type coverage": object_type_coverage,

        # Fractions over ALL model constraints, including inactive ones.
        "Source-activation coverage": _coverage_mean(
            details["source_activated"]
        ),
        "Non-vacuous activation coverage": _coverage_mean(
            details["nonvacuously_activated"]
        ),

        # Equal-weight average of per-constraint presence fractions,
        # restricted to constraints with at least one source activation.
        "Object-presence coverage": _coverage_mean(
            details["object_presence_coverage"]
        ),

        # Equal-weight average over constraints with evaluated bindings.
        # Undefined if none were evaluated.
        "Finite-boundary coverage": _coverage_mean(
            details["finite_boundary_coverage"]
        ),

        # Fraction of positive-minimum constraints with at least one
        # observed matching event. Inactive eligible constraints count 0.
        "Matching-event coverage": _coverage_mean(
            details["matching_event_coverage"]
        ),
    }

    return scores, details


# ---- Reload the current model and logs --------------------------------------
# Use separate names to avoid replacing the confidence cell's variables.

coverage_model = load_confidence_model(OCDECLARE_MODEL)

coverage_idx_input, coverage_types_input = _coverage_index(INPUT_LOG)
coverage_idx_simulated, coverage_types_simulated = _coverage_index(
    SIMULATED_LOG
)

cov_input, det_input = model_coverage(
    coverage_idx_input,
    coverage_types_input,
    coverage_model,
)

cov_sim, det_sim = model_coverage(
    coverage_idx_simulated,
    coverage_types_simulated,
    coverage_model,
)

# Numeric scores and detailed diagnostics remain available for later cells.
coverage_table = pd.DataFrame({
    "input": cov_input,
    "simulated": cov_sim,
})


# ---- Print category scores only --------------------------------------------

def _coverage_format(value):
    return "undefined" if pd.isna(value) else f"{value:.2%}"


print(f"{'Coverage category':36} {'Input':>12} {'Simulated':>12}")
for category in coverage_table.index:
    print(
        f"{category:36} "
        f"{_coverage_format(cov_input[category]):>12} "
        f"{_coverage_format(cov_sim[category]):>12}"
    )

Coverage category                           Input    Simulated
Activity coverage                         100.00%       92.86%
Object-type coverage                      100.00%      100.00%
Source-activation coverage                100.00%       88.57%
Non-vacuous activation coverage           100.00%       88.57%
Object-presence coverage                   99.62%      100.00%
Finite-boundary coverage                   97.14%       80.65%
Matching-event coverage                   100.00%       71.43%
